# Project Bilanx: Cryptographic Validation & Range Proof Benchmark

## 1. Mathematical Formulation
Project Bilanx implements a non-interactive range proof of age compliance on **NIST P-256 (secp256r1)**.

### Pedersen Commitment Scheme
Given generator $G$ and Nothing-Up-My-Sleeve (NUMS) generator $H = \text{hashToCurve}(\text{"Bilanx NUMS H Generator"})$:
$$C = x \cdot G + r \cdot H$$
where:
* $x \in \mathbb{Z}_q$ is the private age value.
* $r \in \mathbb{Z}_q$ is a cryptographically secure random blinding scalar.

Point multiplication is `@noble/curves` `Point.multiply` / `Point.add`, then `toBytes(true)`. It is not WebCrypto.

### Base-2 Sigma Range Proof ($\Delta \in [0, 255]$)
The statement implemented in `src/lib/crypto/zkp/range_b2.ts` is $0 \le x - T < 256$ for the threshold byte $T$ inside the proof. `generateProof` defaults $T$ to 18. `POST /api/v1/age-gate/verify` does not force that byte to 18.

1. Compute $\Delta = x - T = \sum_{i=0}^7 b_i \cdot 2^i$ where each bit $b_i \in \{0, 1\}$.
2. For each bit $i \in \{0, \dots, 7\}$, commit to $b_i$:
   $$C_i = b_i \cdot G + r_i \cdot H$$
3. Prove $b_i \in \{0, 1\}$ using a disjunctive sigma protocol.
4. The verifier checks the sum, matching `verifyBitwiseRangeProof`:
   $$C - T \cdot G - \sum_{i=0}^7 2^i \cdot C_i = r_{diff} \cdot H$$
   where $r_{diff} = r - \sum_{i=0}^7 2^i r_i$.


In [ ]:
# 2. Canonical Binary Wire Format Specification (1,355 Bytes)
# Offset | Field                      | Size (bytes)
# -------+----------------------------+-------------
# 0      | Protocol Version (0x01)    | 1
# 1      | Threshold byte (prover-chosen; demo default 18) | 1
# 2..34  | Master Commitment C        | 33 (compressed P-256)
# 35..   | 8 Bit Proofs:              | 8 * 161 = 1288
#        |   - C_i                    | 33
#        |   - c0, c1 challenges      | 32 + 32 = 64
#        |   - s0, s1 responses       | 32 + 32 = 64
# 1323.. | Blinding Difference r_diff | 32
# Total  |                            | 1355 bytes

EXPECTED_PAYLOAD_SIZE = 1 + 1 + 33 + (8 * (33 + 32 + 32 + 32 + 32)) + 32
print(f"Calculated Wire Payload Size: {EXPECTED_PAYLOAD_SIZE} bytes")
assert EXPECTED_PAYLOAD_SIZE == 1355, "Payload size mismatch!"


## 3. What the code actually records

Names match `useBilanxDualKeyValidation.real.ts` and `tests/b2_acceptance_harness.ts`. Figures are from `npm test` on 6 October 2026. KPI-02 is not the range-proof verifier.

| ID | What the code records | This run | Notes |
| :--- | :--- | :---: | :--- |
| **KPI-01** | Whole dual-key handshake latency, including the server round trip | 116–226 ms with the 80–250 ms dev delay; about 5–7 ms when the test server sets `isTest` | Not ZKP prove time. |
| **KPI-02** | `coreCryptoTime`: local WebCrypto import, hash, sign, and verify inside the handshake | about 0.6–9 ms | Not NIZKP verify. |
| **B2 prove** | Harness prove of age 18 | **204.79 ms** | One sample. |
| **B2 verify** | Same harness, `verifyProof` | **235.18 ms** | One sample. |
| **Pilot clearance** | Swiss adult row `2.1i` | **1084 ms** | Under the 1,500 ms check in that test. `isTest` skips the artificial challenge delay. |
| **Wire size** | Encoded range proof | **1,355 bytes** | T11, T13, T14. |


In [ ]:
# Measured harness sample from 6 October 2026. This cell does not call the verifier.
benchmarks = {
    "measured": {
        "name": "Bilanx Base-2 ZKP",
        "payload_bytes": 1355,
        "prove_ms": 204.79,
        "verify_ms": 235.18,
    },
    "not_measured": [
        "W3C VC (JSON-LD)",
        "eIDAS SAML 2.0 (Full)",
    ],
}

m = benchmarks["measured"]
print(f"Measured: {m['name']}: {m['payload_bytes']} bytes, prove {m['prove_ms']} ms, verify {m['verify_ms']} ms")
print("Not measured in this tree:", ", ".join(benchmarks["not_measured"]))
